In [ ]:
# Notebook 3/9 — Retracted OA PDF Downloader (Other Sources Top-Up)
#
# Purpose: This notebook downloads additional OA PDFs for retracted articles that remain missing after
# earlier download stages (e.g., Unpaywall and CORE). It uses Semantic Scholar, OpenAlex and Europe PMC.
#
# What this notebook does
# 1) Loads the DOI list (code, doi) from Google Drive.
# 2) Detects which PDFs are already present in the output directory.
# 3) For missing cases, searches alternative sources for candidate PDF URLs.
# 4) Downloads PDFs using a safe-write approach (.part then rename) and validates the PDF signature.
# 5) Uses a persistent log to avoid repeating prior attempts across reruns.
# 6) Uses a search cache to avoid repeating expensive searches across reruns.
#
# Inputs:
# - /content/drive/MyDrive/Dissertation/retracted_dois.csv
#   Required columns: code ; doi
#
# Outputs:
# - PDFs saved to: /content/drive/MyDrive/Dissertation/oa_pdfs_all/<code>.pdf
# - Log saved to:  /content/drive/MyDrive/Dissertation/retracted_download_log_topup.csv
#
# Execution:
# - Run top-to-bottom.
# - Reruns are resume-safe: the log and the search cache prevent duplicated work.
#
# Notes:
# - This notebook is designed as a “top-up” stage in the overall pipeline.
# - This notebook is part of a larger, multi-stage data acquisition and
# analysis pipeline and is designed to be fully reproducible.

In [ ]:
# 1) Mount Google Drive + Install dependencies

from google.colab import drive
drive.mount("/content/drive")

!pip -q install pandas requests tqdm tenacity

In [ ]:
# 2) Imports

import os, re, time, json, csv, requests
import pandas as pd
from tqdm.auto import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

In [ ]:
# 3) Configuration

DISSERTATION_DIR = "/content/drive/MyDrive/Dissertation"
INPUT_CSV = os.path.join(DISSERTATION_DIR, "retracted_dois.csv")
OUT_DIR = os.path.join(DISSERTATION_DIR, "oa_pdfs_all")

LOG_PATH = os.path.join(DISSERTATION_DIR, "retracted_download_log_topup.csv")

OPENALEX_WORKS = "https://api.openalex.org/works"
S2_API = "https://api.semanticscholar.org/graph/v1/paper/DOI:"
EUROPE_PMC = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"

os.makedirs(OUT_DIR, exist_ok=True)

MIN_VALID_FILESIZE = 10_000
SLEEP_S = 0.25

print("Input:", INPUT_CSV)
print("Output:", OUT_DIR)
print("Log:", LOG_PATH)

In [ ]:
4) Helpers

def normalize_doi(d):
    if pd.isna(d):
        return None
    s = str(d).strip()
    s = s.replace("https://doi.org/", "").replace("http://doi.org/", "")
    s = s.strip().lower()
    return s if s else None

def sanitize_filename(s):
    s = str(s).strip()
    s = re.sub(r"[^\w\-. ]+", "_", s)
    s = re.sub(r"\s+", "_", s)
    return s[:180]

def is_good_pdf(path):
    if not os.path.exists(path):
        return False
    if os.path.getsize(path) < MIN_VALID_FILESIZE:
        return False
    try:
        with open(path, "rb") as f:
            return f.read(5) == b"%PDF-"
    except Exception:
        return False

@retry(reraise=True, stop=stop_after_attempt(5),
       wait=wait_exponential(multiplier=1, min=1, max=20),
       retry=retry_if_exception_type(requests.RequestException))
def http_get(url, headers=None, timeout=60, stream=True):
    return requests.get(
        url,
        headers=headers or {},
        timeout=timeout,
        stream=stream,
        allow_redirects=True
    )

def download_pdf(url, out_path):
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/pdf,application/octet-stream;q=0.9,*/*;q=0.8",
    }
    tmp = out_path + ".part"
    r = http_get(url, headers=headers, timeout=90, stream=True)
    r.raise_for_status()

    with open(tmp, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 256):
            if chunk:
                f.write(chunk)

    with open(tmp, "rb") as f:
        if f.read(5) != b"%PDF-":
            try:
                os.remove(tmp)
            except Exception:
                pass
            return False, "Not a PDF"

    os.replace(tmp, out_path)
    return True, "OK"

In [ ]:
# 5)  OA resolvers

@retry(reraise=True, stop=stop_after_attempt(5),
       wait=wait_exponential(multiplier=1, min=1, max=20),
       retry=retry_if_exception_type(requests.RequestException))
def semantic_scholar(doi):
    fields = "openAccessPdf,externalIds,url"
    r = requests.get(f"{S2_API}{doi}", params={"fields": fields}, timeout=30)
    if r.status_code == 404:
        return {}
    r.raise_for_status()
    return r.json()

def semantic_scholar_pdf_url(j):
    if not j:
        return None
    oap = j.get("openAccessPdf") or {}
    return oap.get("url")

@retry(reraise=True, stop=stop_after_attempt(5),
       wait=wait_exponential(multiplier=1, min=1, max=20),
       retry=retry_if_exception_type(requests.RequestException))
def openalex_work_by_doi(doi):
    doi_url = f"https://doi.org/{doi}"
    r = requests.get(f"{OPENALEX_WORKS}/{requests.utils.quote(doi_url, safe='')}", timeout=30)
    if r.status_code == 404:
        return {}
    r.raise_for_status()
    return r.json()

def openalex_candidate_urls(j):
    if not j:
        return []
    urls = []
    pl = j.get("primary_location") or {}
    if pl.get("pdf_url"):
        urls.append(pl["pdf_url"])

    oa = j.get("open_access") or {}
    if oa.get("oa_url"):
        urls.append(oa["oa_url"])

    for loc in (j.get("locations") or [])[:10]:
        if isinstance(loc, dict):
            if loc.get("pdf_url"):
                urls.append(loc["pdf_url"])
            if loc.get("landing_page_url"):
                urls.append(loc["landing_page_url"])

    # De-dupe preserve order
    seen = set()
    out = []
    for u in urls:
        if u and u not in seen:
            seen.add(u)
            out.append(u)
    return out

@retry(reraise=True, stop=stop_after_attempt(5),
       wait=wait_exponential(multiplier=1, min=1, max=20),
       retry=retry_if_exception_type(requests.RequestException))
def europe_pmc_by_doi(doi):
    q = f'DOI:"{doi}"'
    r = requests.get(EUROPE_PMC, params={"query": q, "format": "json", "pageSize": 1}, timeout=30)
    r.raise_for_status()
    return r.json()

def pmc_pdf_url_from_europe_pmc(j):
    try:
        hits = j.get("resultList", {}).get("result", [])
        if not hits:
            return None
        pmcid = hits[0].get("pmcid")
        if not pmcid:
            return None
        return f"https://pmc.ncbi.nlm.nih.gov/articles/{pmcid}/pdf/"
    except Exception:
        return None

In [ ]:
# 6) Load input & compute missing

df = pd.read_csv(INPUT_CSV, sep=";", encoding="utf-8-sig", dtype={"code": "string", "doi": "string"})
df.columns = [c.strip().replace("\ufeff", "") for c in df.columns]

if "code" not in df.columns or "doi" not in df.columns:
    raise ValueError(f"Expected columns 'code' and 'doi'. Found: {df.columns.tolist()}")

df["code"] = df["code"].astype("string").str.strip()
df["doi"] = df["doi"].apply(normalize_doi)

existing_stems = set(
    os.path.splitext(f)[0].strip()
    for f in os.listdir(OUT_DIR)
    if f.lower().endswith(".pdf")
)

print("PDFs found in folder:", len(existing_stems))
print("Example stems:", sorted(list(existing_stems))[:10])

df["code_num"] = df["code"].str.extract(r"(\d+)")[0].fillna("").astype("string")

df["already_in_folder"] = df["code_num"].isin(existing_stems)
missing = df[~df["already_in_folder"]].copy()

print("Total rows:", len(df))
print("Already in folder (fixed):", int(df["already_in_folder"].sum()))
print("Missing:", len(missing))

print("\nFirst 10 codes:", df["code"].head(10).tolist())
print("First 10 code_num:", df["code_num"].head(10).tolist())

In [ ]:
# 7) Prepare / load log (backward-compatible, resume-safe)

LOG_COLS = ["code","code_num","doi","status","source","pdf_url","filepath","message"]

if not os.path.exists(LOG_PATH):
    pd.DataFrame(columns=LOG_COLS).to_csv(LOG_PATH, index=False)

log = pd.read_csv(LOG_PATH)

# Ensure expected cols exist
for c in LOG_COLS:
    if c not in log.columns:
        log[c] = ""

# Backfill code_num in log if missing
def _to_code_num(x):
    s = str(x).strip()
    m = re.search(r"(\d+)", s)
    return m.group(1) if m else ""

log["code"] = log["code"].astype(str).str.strip()
log["code_num"] = log["code_num"].astype(str).str.strip()
mask_missing = (log["code_num"] == "") | (log["code_num"].str.lower().isin(["nan", "none"]))
log.loc[mask_missing, "code_num"] = log.loc[mask_missing, "code"].apply(_to_code_num)

# Treat "searched" as a durable checkpoint too
treated_codes_logged = set(
    log.loc[log["status"].isin(["downloaded", "failed", "searched"]), "code_num"]
       .astype(str).str.strip().tolist()
)

downloaded_codes_logged = set(
    log.loc[log["status"] == "downloaded", "code_num"]
       .astype(str).str.strip().tolist()
)

# Build a cache of candidates from prior "searched" rows so reruns do NOT re-query APIs
searched_cache = {}

if "status" in log.columns and "message" in log.columns:
    searched_rows = log[log["status"] == "searched"][["code_num", "message"]].copy()
    for _, r in searched_rows.iterrows():
        cn = str(r["code_num"]).strip()
        msg = r["message"]
        if not isinstance(msg, str) or not msg:
            continue
        try:
            obj = json.loads(msg)
            cand = obj.get("candidates", [])
            if isinstance(cand, list) and cand:
                # keep the most recent candidates we saw for that code_num
                searched_cache[cn] = [(a, b) for a, b in cand if a and b]
        except Exception:
            # ignore older free-text messages
            pass

print("Already logged attempted (downloaded+failed+searched):", len(treated_codes_logged))
print("Already logged downloaded:", len(downloaded_codes_logged))
print("Cached candidate lists from prior searches:", len(searched_cache))

In [ ]:
# 8) Downloader loop (CORE-style bar + postfix + Saved)

print("Step 6 starting...", flush=True)

# Log writing: append-one-row (resume-safe)
LOG_FLUSH_EVERY = 1
_append_count = 0

def append_log_row(row_dict):
    global _append_count
    file_exists = os.path.exists(LOG_PATH) and os.path.getsize(LOG_PATH) > 0
    row = {c: row_dict.get(c, "") for c in LOG_COLS}

    with open(LOG_PATH, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=LOG_COLS)
        if not file_exists:
            w.writeheader()
        w.writerow(row)

        _append_count += 1
        if LOG_FLUSH_EVERY and (_append_count % LOG_FLUSH_EVERY == 0):
            try:
                f.flush()
                os.fsync(f.fileno())
            except Exception:
                pass


downloaded_n = 0
skipped_exists = 0
skipped_logged = 0
failed_no_url = 0
failed_dl = 0
removed_bad = 0

# FAST disk truth: size-only scan
pdf_files = [f for f in os.listdir(OUT_DIR) if f.lower().endswith(".pdf")]
print(f"Scanning existing PDFs in folder: {len(pdf_files)} files", flush=True)

existing_ok_stems = set()
for fn in tqdm(pdf_files, desc="Scanning folder", unit="pdf", dynamic_ncols=True):
    stem = os.path.splitext(fn)[0].strip()
    full = os.path.join(OUT_DIR, fn)
    try:
        if os.path.getsize(full) >= MIN_VALID_FILESIZE:
            existing_ok_stems.add(stem)
    except Exception:
        pass

print(f"Folder scan done. Usable PDFs by size: {len(existing_ok_stems)}", flush=True)

def is_good_pdf_strict(path):
    if not os.path.exists(path):
        return False
    if os.path.getsize(path) < MIN_VALID_FILESIZE:
        return False
    try:
        with open(path, "rb") as f:
            return f.read(5) == b"%PDF-"
    except Exception:
        return False

print(f"Processing missing rows: {len(missing)}", flush=True)
pbar = tqdm(total=len(missing), desc="OA missing cases", unit="paper", dynamic_ncols=True, leave=True)

for _, row in missing.iterrows():
    code = str(row["code"]).strip()
    code_num = str(row.get("code_num", "")).strip()
    doi = row["doi"]

    if not code_num or code_num.lower() in ("nan", "none"):
        m = re.search(r"(\d+)", code)
        code_num = m.group(1) if m else code

    out_path = os.path.join(OUT_DIR, f"{code_num}.pdf")
    legacy_path = os.path.join(OUT_DIR, f"{sanitize_filename(code)}.pdf")

    # 1) Skip if file already exists and seems OK by size
    if (code_num in existing_ok_stems) or \
       (os.path.exists(out_path) and os.path.getsize(out_path) >= MIN_VALID_FILESIZE) or \
       (os.path.exists(legacy_path) and os.path.getsize(legacy_path) >= MIN_VALID_FILESIZE):
        skipped_exists += 1
        existing_ok_stems.add(code_num)
        pbar.set_postfix(downloaded=downloaded_n, skip_exists=skipped_exists, skip_logged=skipped_logged,
                         fail_no_url=failed_no_url, fail_dl=failed_dl, rm_bad=removed_bad)
        pbar.update(1)
        continue

    # Remove small/bad remnants so we can retry cleanly
    if os.path.exists(out_path) and os.path.getsize(out_path) < MIN_VALID_FILESIZE:
        try:
            os.remove(out_path); removed_bad += 1
        except Exception:
            pass
    if os.path.exists(legacy_path) and os.path.getsize(legacy_path) < MIN_VALID_FILESIZE:
        try:
            os.remove(legacy_path); removed_bad += 1
        except Exception:
            pass

    # 2) Resume-safe skip: if already logged as searched/ failed/ downloaded
    if code_num in treated_codes_logged and (code_num not in searched_cache):
        skipped_logged += 1
        pbar.set_postfix(downloaded=downloaded_n, skip_exists=skipped_exists, skip_logged=skipped_logged,
                         fail_no_url=failed_no_url, fail_dl=failed_dl, rm_bad=removed_bad)
        pbar.update(1)
        continue

    # 3) Missing DOI
    if not doi:
        append_log_row({
            "code": code, "code_num": code_num, "doi": doi,
            "status": "failed", "source": "", "pdf_url": "",
            "filepath": out_path, "message": "Missing DOI",
        })
        treated_codes_logged.add(code_num)
        failed_no_url += 1
        pbar.set_postfix(downloaded=downloaded_n, skip_exists=skipped_exists, skip_logged=skipped_logged,
                         fail_no_url=failed_no_url, fail_dl=failed_dl, rm_bad=removed_bad)
        pbar.update(1)
        continue

    # 4) Get candidates:
    #    - Prefer cached candidates from a prior run (no re-search)
    #    - Otherwise call APIs and immediately log "searched" with the candidates JSON
    if code_num in searched_cache and searched_cache[code_num]:
        cand2 = searched_cache[code_num]
    else:
        candidates = []

        try:
            sj = semantic_scholar(doi)
            su = semantic_scholar_pdf_url(sj)
            if su:
                candidates.append(("semantic_scholar", su))
        except Exception:
            pass

        try:
            oj = openalex_work_by_doi(doi)
            for u in openalex_candidate_urls(oj):
                if u:
                    candidates.append(("openalex", u))
        except Exception:
            pass

        try:
            ej = europe_pmc_by_doi(doi)
            pu = pmc_pdf_url_from_europe_pmc(ej)
            if pu:
                candidates.append(("europepmc", pu))
        except Exception:
            pass

        # De-dupe preserve order
        seen = set()
        cand2 = []
        for src, u in candidates:
            if u and u not in seen:
                seen.add(u)
                cand2.append((src, u))

        # Durable Checkpoint: log the search result immediately
        append_log_row({
            "code": code,
            "code_num": code_num,
            "doi": doi,
            "status": "searched",
            "source": "mix",
            "pdf_url": "",
            "filepath": out_path,
            "message": json.dumps({"candidates": cand2}, ensure_ascii=False),
        })
        searched_cache[code_num] = cand2
        treated_codes_logged.add(code_num)

    if not cand2:
        # Search happened (or was cached) but no candidates
        append_log_row({
            "code": code,
            "code_num": code_num,
            "doi": doi,
            "status": "failed",
            "source": "",
            "pdf_url": "",
            "filepath": out_path,
            "message": "No candidate OA URLs (S2/OpenAlex/EuropePMC)",
        })
        failed_no_url += 1
        pbar.set_postfix(downloaded=downloaded_n, skip_exists=skipped_exists, skip_logged=skipped_logged,
                         fail_no_url=failed_no_url, fail_dl=failed_dl, rm_bad=removed_bad)
        pbar.update(1)
        continue

    # 5) Try candidates in order
    success = False
    last_msg = ""
    last_url = ""
    last_src = ""

    for src, url in cand2:
        try:
            ok, msg = download_pdf(url, out_path)
            if ok and is_good_pdf_strict(out_path):
                append_log_row({
                    "code": code, "code_num": code_num, "doi": doi,
                    "status": "downloaded", "source": src, "pdf_url": url,
                    "filepath": out_path, "message": msg,
                })
                existing_ok_stems.add(code_num)
                downloaded_n += 1
                print(f"Saved: {out_path}", flush=True)
                success = True
                break
            else:
                try:
                    if os.path.exists(out_path):
                        os.remove(out_path)
                        removed_bad += 1
                except Exception:
                    pass
                last_msg = msg if msg else "Downloaded file invalid or too small"
                last_url = url
                last_src = src
        except Exception as e:
            last_msg = str(e)
            last_url = url
            last_src = src

        time.sleep(SLEEP_S)

    if not success:
        append_log_row({
            "code": code, "code_num": code_num, "doi": doi,
            "status": "failed", "source": last_src, "pdf_url": last_url,
            "filepath": out_path, "message": last_msg,
        })
        failed_dl += 1

    pbar.set_postfix(downloaded=downloaded_n, skip_exists=skipped_exists, skip_logged=skipped_logged,
                     fail_no_url=failed_no_url, fail_dl=failed_dl, rm_bad=removed_bad)
    pbar.update(1)
    time.sleep(SLEEP_S)

pbar.close()

print("\n--- SUMMARY (RESUME-SAFE + SEARCH-CACHED) ---", flush=True)
print("Downloaded this run:", downloaded_n, flush=True)
print("Skipped (exists/size-ok):", skipped_exists, flush=True)
print("Skipped (already attempted non-search):", skipped_logged, flush=True)
print("Removed small/broken PDFs:", removed_bad, flush=True)
print("Failed (no candidate URL / missing DOI):", failed_no_url, flush=True)
print("Failed (download/invalid):", failed_dl, flush=True)
print("Search-cache size:", len(searched_cache), flush=True)
print("Log:", LOG_PATH, flush=True)
print("Folder:", OUT_DIR, flush=True)